# Sentence Transformer only Approach

#### Models
- Bi-Encoder: `nomic-ai/nomic-embed-text-v1.5`
- Cross-Encoder: `cross-encoder/ettin-reranker-1b-v1`

#### Functionality
- Search Wikipedia RAG for relevant articles for question and options
- Find top-k relevant sections using Bi-Encoder
- Re-rank relevant sections with Cross-Encoder
- Infer answer based on highest Cross-Encoder score

## Setup environment

In [ ]:
!pip install torch sentence-transformers wikipedia-api requests librosa openai-whisper pandas openpyxl "transformers>=5.2.0"

In [ ]:
import torch
import os
import time
import re
import io
import threading
import dataclasses
import numpy as np
import pandas as pd
import requests
import pprint
from google.colab import userdata
from collections import defaultdict
from datetime import datetime
from abc import ABC, abstractmethod
from typing import Any, List, Optional, Dict, Union
from enum import Enum
from dataclasses import dataclass, field
from uuid import uuid4
from urllib.parse import urljoin
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import wikipediaapi
torch.set_float32_matmul_precision('high')

## Setup Millionaire client

This self-contained notebook includes the relevant classes and functions of the Millionaire client. These classes are unmodified from the provided code.

In [5]:
class MillionaireError(Exception):
    def __init__(self, message: str, status_code: Optional[int] = None, response_data: Optional[dict] = None):
        super().__init__(message)
        self.message = message
        self.status_code = status_code
        self.response_data = response_data or {}
    def __str__(self):
        return f"[{self.status_code}] {self.message}" if self.status_code else self.message

class AuthenticationError(MillionaireError): pass
class GameError(MillionaireError): pass
class TimeoutError(MillionaireError): pass
class ValidationError(MillionaireError): pass
class NotFoundError(MillionaireError): pass
class ServerError(MillionaireError): pass
class RateLimitError(MillionaireError): pass

class GameStatus(Enum):
    IN_PROGRESS = "in_progress"
    COMPLETED = "completed"
    FAILED = "failed"
    TIMEOUT = "timeout"

@dataclass
class Option:
    id: int
    text: Optional[str] = None
    @classmethod
    def from_dict(cls, data: dict) -> "Option":
        return cls(id=data["id"], text=data.get("text"))

@dataclass
class Question:
    id: int
    text: Optional[str] = None
    options: List[Option] = field(default_factory=list)
    level: int = 0
    @classmethod
    def from_dict(cls, data: dict) -> "Question":
        return cls(id=data["id"], text=data.get("text"),
                   options=[Option.from_dict(opt) for opt in data.get("options", [])],
                   level=data.get("level", 0))

@dataclass
class MoneyLevel:
    level: int
    amount: float
    @classmethod
    def from_dict(cls, data: dict) -> "MoneyLevel":
        return cls(level=data["level"], amount=data["amount"])

@dataclass
class Competition:
    id: int
    name: str
    description: Optional[str] = None
    max_levels: int = 15
    is_infinite: bool = False
    @classmethod
    def from_dict(cls, data: dict) -> "Competition":
        return cls(id=data["id"], name=data["name"], description=data.get("description"),
                   max_levels=data.get("maxLevels", 15), is_infinite=data.get("isInfinite", False))

@dataclass
class GameState:
    session_id: int
    competition: Competition
    status: GameStatus
    earned_amount: float
    current_level: int
    money_pyramid: List[MoneyLevel]
    question_deadline: Optional[datetime] = None
    question: Optional[Question] = None
    mode: str = "text"

    @classmethod
    def from_dict(cls, data: dict) -> "GameState":
        deadline = data.get("questionDeadline")
        if deadline:
            try: deadline = datetime.fromisoformat(deadline.replace("Z", "+00:00"))
            except: deadline = None
        return cls(
            session_id=data.get("sessionId", data.get("id", 0)),
            competition=Competition.from_dict(data["competition"]),
            status=GameStatus(data.get("status", "in_progress")),
            earned_amount=data.get("earnedAmount", 0),
            current_level=data.get("currentLevel", 1),
            money_pyramid=[MoneyLevel.from_dict(ml) for ml in data.get("moneyPyramid", [])],
            question_deadline=deadline,
            question=Question.from_dict(data["question"]) if data.get("question") else None,
            mode=data.get("mode", "text"),
        )
    @property
    def in_progress(self) -> bool: return self.status == GameStatus.IN_PROGRESS
    @property
    def is_game_over(self) -> bool: return self.status in (GameStatus.COMPLETED, GameStatus.FAILED, GameStatus.TIMEOUT)

@dataclass
class AnswerResult:
    correct: Optional[bool] = None
    game_over: bool = False
    earned_amount: float = 0
    timed_out: bool = False
    status: Optional[str] = None
    current_level: Optional[int] = None
    question_deadline: Optional[datetime] = None
    question: Optional[Question] = None
    money_pyramid: List[MoneyLevel] = field(default_factory=list)

    @classmethod
    def from_dict(cls, data: dict) -> "AnswerResult":
        deadline = data.get("questionDeadline")
        if deadline:
            try: deadline = datetime.fromisoformat(deadline.replace("Z", "+00:00"))
            except: deadline = None
        return cls(
            correct=data.get("correct"),
            game_over=data.get("gameOver", False),
            earned_amount=data.get("earnedAmount", 0),
            timed_out=data.get("timedOut", False),
            status=data.get("status"),
            current_level=data.get("currentLevel"),
            question_deadline=deadline,
            question=Question.from_dict(data["question"]) if data.get("question") else None,
            money_pyramid=[MoneyLevel.from_dict(ml) for ml in data.get("moneyPyramid", [])]
        )

class BaseClient:
    def __init__(self, base_url: str, timeout: int = 30):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self._session = requests.Session()

    def set_auth_cookie(self, cookie_value: str):
        self._session.cookies.set("polimillionaire_auth", cookie_value)

    def is_authenticated(self) -> bool:
        return "polimillionaire_auth" in self._session.cookies

    def request(self, method: str, endpoint: str, data=None, params=None, auth_required=True, raw=False) -> Any:
        if auth_required and not self.is_authenticated():
            raise AuthenticationError("Authentication required.")
        url = urljoin(f"{self.base_url}/", endpoint.lstrip("/"))
        try:
            resp = self._session.request(method, url, json=data, params=params, timeout=self.timeout)
            if raw and resp.status_code < 400: return resp.content

            try: data = resp.json() if resp.text else {}
            except: data = {}

            if resp.status_code in (200, 201, 204): return data
            msg = data.get("message", data.get("error", f"HTTP {resp.status_code}"))
            if resp.status_code == 401: raise AuthenticationError(msg)
            if resp.status_code == 404: raise NotFoundError(msg)
            raise MillionaireError(msg, resp.status_code)
        except requests.Timeout: raise MillionaireError("Timeout")
        except requests.ConnectionError: raise MillionaireError("Connection Error")

class GameSession:
    def __init__(self, client: BaseClient, state: GameState):
        self._client = client
        self._state = state
    @property
    def session_id(self) -> int: return self._state.session_id
    @property
    def in_progress(self) -> bool: return self._state.in_progress
    @property
    def current_question(self) -> Optional[Question]: return self._state.question
    @property
    def current_level(self) -> int: return self._state.current_level
    @property
    def mode(self) -> str: return self._state.mode
    @property
    def state(self) -> GameState: return self._state

    def fetch_audio_question(self) -> bytes:
        return self._client.request("GET", f"/api/game/{self.session_id}/audio/question", raw=True)
    def fetch_audio_option_next(self) -> bytes:
        return self._client.request("GET", f"/api/game/{self.session_id}/audio/option/next", raw=True)

    def answer(self, option_id: int) -> AnswerResult:
        res_data = self._client.request("POST", f"/api/game/{self.session_id}/answer", data={"optionId": option_id})
        res = AnswerResult.from_dict(res_data)
        if res.question:
            self._state = GameState.from_dict({
                "sessionId": self.session_id, "competition": self._state.competition.__dict__,
                "status": "in_progress", "earnedAmount": res.earned_amount,
                "currentLevel": res.current_level or self._state.current_level,
                "moneyPyramid": [ml.__dict__ for ml in res.money_pyramid] if res.money_pyramid else [],
                "questionDeadline": res.question_deadline.isoformat() if res.question_deadline else None,
                "mode": self._state.mode, "question": {
                    "id": res.question.id, "text": res.question.text,
                    "options": [{"id": o.id, "text": o.text} for o in res.question.options]
                }
            })
        elif res.game_over:
            if res.timed_out:
                self._state.status = GameStatus.TIMEOUT
            else:
                self._state.status = GameStatus.COMPLETED if res.correct else GameStatus.FAILED
            self._state.earned_amount = res.earned_amount
            self._state.question = None
        return res

class MillionaireClient:
    def __init__(self, base_url: str):
        self._base = BaseClient(base_url)
    def login(self, username, password):
        resp = self._base.request("POST", "/api/auth/login", data={"username": username, "password": password}, auth_required=False)
        return resp
    @property
    def game(self):
        class GameModule:
            def __init__(self, base): self._base = base
            def start(self, competition_id, mode="text"):
                resp = self._base.request("POST", "/api/game/start", data={"competitionId": competition_id, "mode": mode})
                return GameSession(self._base, GameState.from_dict(resp))
        return GameModule(self._base)

## Guesser and Benchmark framework

The notebook contains our team-internal Guesser and Benchmark framework to standardize experiments across team members. It handles everything the interaction with the client as well as monitoring performance and time metrics. Additionally, the base `Guesser` class handles the speech to text conversion. The game mode can be set via the `mode` parameter as either `text` or `speech`.

The speech transcription uses the `whisper` package. Both the `tiny` and the `base` transcription models are supported to suit different inference model approaches.

After the speech synthesis we use multiple `regex` to filter out artifacts like laughs, singing and filler words (ahm, uhhh, ...). Additionally, we filter out transcriptions of *Option A, B, C, D* as they do not provide information and hurt our framework-based inference approach.

In [ ]:
class ApproachType(str, Enum):
    DIRECT_LLM = "direct_llm"
    RAG = "rag"
    HYBRID = "hybrid"

class QuestionOutcome(str, Enum):
    CORRECT = "correct"
    INCORRECT = "incorrect"
    TIMEOUT = "timeout"
    ERROR = "error"

@dataclass
class ExperimentConfig:
    experiment_id: str = field(init=False, default_factory=lambda: uuid4().hex)
    username: str = ""
    notes: str = ""
    approach: Optional[ApproachType] = None
    inference_model: str = ""
    inference_model_size: str = ""
    is_rag: bool = False
    embedding_model: Optional[str] = None
    embedding_model_size: Optional[str] = None
    # General Metrics
    mean_question_accuracy: float = 0.0
    mean_time: float = 0.0
    mean_search_time: float = 0.0
    mean_reasoning_time: float = 0.0
    mean_transcription_time: float = 0.0

    # Theme-specific Metrics
    entertainment_mean_time: float = 0.0
    entertainment_mean_search_time: float = 0.0
    entertainment_mean_reasoning_time: float = 0.0
    entertainment_mean_transcription_time: float = 0.0
    entertainment_mean_question_accuracy: float = 0.0

    ancient_history_mean_time: float = 0.0
    ancient_history_mean_search_time: float = 0.0
    ancient_history_mean_reasoning_time: float = 0.0
    ancient_history_mean_transcription_time: float = 0.0
    ancient_history_mean_question_accuracy: float = 0.0

    science_nature_mean_time: float = 0.0
    science_nature_mean_search_time: float = 0.0
    science_nature_mean_reasoning_time: float = 0.0
    science_nature_mean_transcription_time: float = 0.0
    science_nature_mean_question_accuracy: float = 0.0

    maths_mean_time: float = 0.0
    maths_mean_search_time: float = 0.0
    maths_mean_reasoning_time: float = 0.0
    maths_mean_transcription_time: float = 0.0
    maths_mean_question_accuracy: float = 0.0

    news_mean_time: float = 0.0
    news_mean_search_time: float = 0.0
    news_mean_reasoning_time: float = 0.0
    news_mean_transcription_time: float = 0.0
    news_mean_question_accuracy: float = 0.0

    philosophy_psychology_mean_time: float = 0.0
    philosophy_psychology_mean_search_time: float = 0.0
    philosophy_psychology_mean_reasoning_time: float = 0.0
    philosophy_psychology_mean_transcription_time: float = 0.0
    philosophy_psychology_mean_question_accuracy: float = 0.0

@dataclass
class QuestionResult:
    theme: str = ""
    question_outcome: QuestionOutcome = QuestionOutcome.ERROR
    answer_time: float = 0.0
    search_time: float = 0.0
    reasoning_time: float = 0.0
    transcription_time: float = 0.0
    level: int = 0

class Guesser(ABC):
    """
    Generic interface for all Guessers.
    """
    _whisper_models = {}
    _whisper_lock = threading.Lock()

    def __init__(self, config: ExperimentConfig, mode: str = "text", transcription_model: str = "tiny"):
        if not isinstance(config, ExperimentConfig):
            raise ValueError(f"config must be a {type(ExperimentConfig)}")
        self.config = config
        self.mode = mode
        self.transcription_model_size = transcription_model
        self.search_time: float = 0.0
        self.reasoning_time: float = 0.0
        self.transcription_time: float = 0.0

    def preload(self):
        """Preload models to avoid delays during the first question."""
        if self.mode == "speech":
            _ = self.whisper_model
            print(f"Whisper '{self.transcription_model_size}' model preloaded and ready.")

    @property
    def whisper_model(self):
        """Thread-safe lazy loading of the specified Whisper model."""
        size = self.transcription_model_size
        if size not in Guesser._whisper_models:
            with Guesser._whisper_lock:
                # Double-check pattern to handle concurrent threads
                if size not in Guesser._whisper_models:
                    try:
                        import whisper
                        print(f"Loading Whisper '{size}' model...")
                        Guesser._whisper_models[size] = whisper.load_model(size)
                    except ImportError:
                        print("Error: 'whisper' library not found. Please install it with 'pip install openai-whisper'.")
                        raise
        return Guesser._whisper_models[size]


    def print(self):
        pprint.pprint(self.config)

    @abstractmethod
    def infer_answer(self, question: Question, theme: str, game_session: Any = None) -> int:
        pass

    def format_question_for_llm(self, question: Question) -> str:
        """
        Format question
         """
        prompt_lines = [f"Question: {question.text}\n", "Options:"]

        for index, option in enumerate(question.options):
            prompt_lines.append(f"[{index}] {option.text}")

        return "\n".join(prompt_lines)

    def _transcribe_audio(self, audio_data: bytes) -> str:
        import librosa
        import whisper

        try:
            # transcribe in memory
            audio_np, _ = librosa.load(io.BytesIO(audio_data), sr=16000)

            # trim silence
            audio_np, _ = librosa.effects.trim(audio_np)
            if audio_np.size == 0:
                return ""

            # enforce english language
            audio_padded = whisper.pad_or_trim(audio_np)

            # avoid race conditions
            with Guesser._whisper_lock:
                mel = whisper.log_mel_spectrogram(audio_padded).to(self.whisper_model.device)
                options = whisper.DecodingOptions(language="en", fp16=False)
                result = whisper.decode(self.whisper_model, mel, options)
                text = result.text

            return self._post_process_text(text)
        except Exception as e:
            print(f"Warning: Transcription failed: {e}")
            return ""

    def _post_process_text(self, text: str) -> str:
        text = text.strip()

        # trim transcription
        if (text.startswith('(') and text.endswith(')')) or (text.startswith('[') and text.endswith(']')):
            text = text[1:-1]
        text = re.sub(r'[\[\(].*?[\]\)]', '', text)

        # remove artifact patterns
        patterns = [
            r'(\b\w+)\b([- ]?\1\b){2,}',                                         # Repeated words (3+ times)
            r'^(?:options?|[tp]op\w*).*?\b(?:[a-d]|see|sea)\b',                 # Start-of-sentence prefix mishears
            r'\b(?:options?|topst?ion|topson|topption|pops|topsynd)\w*\b(?:\s+(?:and\s+)?\w+)?', # Stray false options
            r'\b(?:duh|uh|um|uhm|ah|ha|he|hi|ho|hm+)\b(?:\s+(?:duh|uh|um|uhm|ah|ha|he|hi|ho|hm+)\b)*' # Fillers & stuttering
        ]

        for pattern in patterns:
            text = re.sub(pattern, '', text, flags=re.IGNORECASE)

        # clean up punctuation
        text = re.sub(r'\s+', ' ', text)
        return re.sub(r'^[\s,.;:?!-]+|[\s,.;:?!-]+$', '', text).strip()

    def get_speech_question(self, game_session: Any) -> Question:
        if game_session is None:
            raise ValueError("game_session is required for speech mode")

        print("Speech mode active: Fetching and transcribing audio...")
        self.transcription_time = 0.0

        q_audio = game_session.fetch_audio_question()
        results = {}
        threads = []

        def transcription_task(key, audio_bytes):
            results[key] = self._transcribe_audio(audio_bytes)

        # start transcription upon reception
        t_q = threading.Thread(target=transcription_task, args=("q", q_audio))
        t_q.start()
        threads.append(t_q)

        options_ids = []

        for i in range(4):
            opt_audio = game_session.fetch_audio_option_next()

            t_opt = threading.Thread(target=transcription_task, args=(f"opt_{i}", opt_audio))
            t_opt.start()
            threads.append(t_opt)

            # preserve option number
            if game_session.current_question and i < len(game_session.current_question.options):
                options_ids.append(game_session.current_question.options[i].id)
            else:
                options_ids.append(i)

        # start timer after last option is pulled
        start_transcription_tracking = time.time()

        print("  Waiting for transcriptions to finish...")
        for t in threads:
            t.join()

        self.transcription_time = time.time() - start_transcription_tracking

        transcribed_options = []
        for i in range(4):
            text = results.get(f"opt_{i}", "")
            transcribed_options.append(Option(id=options_ids[i], text=text))

        return Question(
            id=game_session.current_question.id if game_session.current_question else 0,
            text=results.get("q", ""),
            options=transcribed_options,
            level=game_session.current_level
        )


class Game:
    def __init__(self, client : MillionaireClient, guesser : Guesser, competition_id=1):
        self.guesser = guesser
        self.client = client
        self.game = None
        self.competition_id = competition_id
        self.results = []

    def play_game(self):
        mode = getattr(self.guesser, "mode", "text")

        # Load transcription model before game start
        if hasattr(self.guesser, "preload"):
            print("Preloading models...")
            self.guesser.preload()

        self.game = self.client.game.start(competition_id=self.competition_id, mode=mode)
        print(f"Started game in competition: {self.game.state.competition.name} (Mode: {mode})")

        while self.game.in_progress:
            question = self.game.current_question
            if not question:
                break

            print(f"\n--- Level {self.game.current_level} ---")

            transcription_time = 0.0
            if mode == "speech":
                question = self.guesser.get_speech_question(self.game)
                transcription_time = getattr(self.guesser, 'transcription_time', 0.0)

            print(f"Q: {question.text}")
            for i, opt in enumerate(question.options):
                print(f"  [{i}] {opt.text}")

            duration = 0.0
            start_t = time.time()
            try:
                print("\nGuesser is thinking...")
                answer_id = self.guesser.infer_answer(question, self.game.state.competition.name, game_session=self.game)
                duration = time.time() - start_t

                # Retrieve separated metrics if available
                search_time = getattr(self.guesser, 'search_time', 0.0)
                reasoning_time = getattr(self.guesser, 'reasoning_time', 0.0)

                print(f"Guesser chose option: {answer_id} (Time: {duration:.2f}s, Transcription: {transcription_time:.2f}s, Search: {search_time:.2f}s, Reasoning: {reasoning_time:.2f}s)")
            except Exception as e:
                duration = time.time() - start_t
                search_time = getattr(self.guesser, 'search_time', 0.0)
                reasoning_time = getattr(self.guesser, 'reasoning_time', 0.0)
                print(f"Inference error: {e} (Time: {duration:.2f}s, Transcription: {transcription_time:.2f}s, Search: {search_time:.2f}s, Reasoning: {reasoning_time:.2f}s)")
                self.results.append(QuestionResult(
                    theme=self.game.state.competition.name,
                    question_outcome=QuestionOutcome.ERROR,
                    answer_time=duration,
                    search_time=search_time,
                    reasoning_time=reasoning_time,
                    transcription_time=transcription_time,
                    level=self.game.state.current_level,
                ))
                break

            # Submit answer
            result = self.game.answer(answer_id)

            if result.correct:
                print(" CORRECT!")
                question_result = QuestionResult(
                    theme=self.game.state.competition.name,
                    question_outcome=QuestionOutcome.CORRECT,
                    answer_time=duration,
                    search_time=search_time,
                    reasoning_time=reasoning_time,
                    transcription_time=transcription_time,
                    level=self.game.state.current_level,
                )
            elif result.timed_out:
                question_result = QuestionResult(
                    theme=self.game.state.competition.name,
                    question_outcome=QuestionOutcome.TIMEOUT,
                    answer_time=duration,
                    search_time=search_time,
                    reasoning_time=reasoning_time,
                    transcription_time=transcription_time,
                    level=self.game.state.current_level,
                )
            else:
                question_result = QuestionResult(
                    theme=self.game.state.competition.name,
                    question_outcome=QuestionOutcome.INCORRECT,
                    answer_time=duration,
                    search_time=search_time,
                    reasoning_time=reasoning_time,
                    transcription_time=transcription_time,
                    level=self.game.state.current_level,
                )
            self.results.append(question_result)
            if result.game_over:
                status = result.status
                if status is None:
                    # Fallback for when result.status is None (e.g. incorrect answer causing game over)
                    status_text = "INCORRECT" if result.correct is False else "FINISHED"
                else:
                    status_text = status.value if hasattr(status, "value") else str(status)

                if result.correct:
                    print(f"CONGRATULATIONS! Final earnings: ${result.earned_amount:,.2f}")
                else:
                    print(f"GAME OVER! Result: {status_text.upper()}")
                break

    def get_game_results(self):
        return self.results

class Analysis:
    def __init__(self, results: list[QuestionResult], experiment: ExperimentConfig):
        self.results = results
        self.experiment = experiment

    def calculate_experiments_metrics(self) -> None:
        if not self.results:
            return

        total_time = 0.0
        total_search_time = 0.0
        total_reasoning_time = 0.0
        total_transcription_time = 0.0
        total_correct = 0
        total_count = len(self.results)

        theme_stats = defaultdict(lambda: {
            "total_time": 0.0,
            "total_search_time": 0.0,
            "total_reasoning_time": 0.0,
            "total_transcription_time": 0.0,
            "correct_count": 0,
            "count": 0
        })

        for res in self.results:
            is_correct = 1 if res.question_outcome.name == "CORRECT" else 0
            total_time += res.answer_time
            total_search_time += res.search_time
            total_reasoning_time += res.reasoning_time
            total_transcription_time += res.transcription_time
            total_correct += is_correct

            theme_stats[res.theme]["total_time"] += res.answer_time
            theme_stats[res.theme]["total_search_time"] += res.search_time
            theme_stats[res.theme]["total_reasoning_time"] += res.reasoning_time
            theme_stats[res.theme]["total_transcription_time"] += res.transcription_time
            theme_stats[res.theme]["correct_count"] += is_correct
            theme_stats[res.theme]["count"] += 1

        self.experiment.mean_time = total_time / total_count
        self.experiment.mean_search_time = total_search_time / total_count
        self.experiment.mean_reasoning_time = total_reasoning_time / total_count
        self.experiment.mean_transcription_time = total_transcription_time / total_count
        self.experiment.mean_question_accuracy = total_correct / total_count

        theme_map = {
            "Entertainment": "entertainment",
            "Ancient History and Politics": "ancient_history",
            "Science and Nature": "science_nature",
            "Maths": "maths",
            "News": "news",
            "Philosophy and Psychology": "philosophy_psychology",
        }

        for theme_name, stats in theme_stats.items():
            prefix = theme_map.get(theme_name)

            if prefix and stats["count"] > 0:
                avg_time = stats["total_time"] / stats["count"]
                avg_search = stats["total_search_time"] / stats["count"]
                avg_reasoning = stats["total_reasoning_time"] / stats["count"]
                avg_transcription = stats["total_transcription_time"] / stats["count"]
                avg_acc = stats["correct_count"] / stats["count"]

                # Attribute names
                attr_time = f"{prefix}_mean_time"
                attr_search = f"{prefix}_mean_search_time"
                attr_reasoning = f"{prefix}_mean_reasoning_time"
                attr_transcription = f"{prefix}_mean_transcription_time"
                attr_acc = f"{prefix}_mean_question_accuracy"

                if hasattr(self.experiment, attr_time):
                    setattr(self.experiment, attr_time, avg_time)
                if hasattr(self.experiment, attr_search):
                    setattr(self.experiment, attr_search, avg_search)
                if hasattr(self.experiment, attr_reasoning):
                    setattr(self.experiment, attr_reasoning, avg_reasoning)
                if hasattr(self.experiment, attr_transcription):
                    setattr(self.experiment, attr_transcription, avg_transcription)
                if hasattr(self.experiment, attr_acc):
                    setattr(self.experiment, attr_acc, avg_acc)

class Benchmark:
    def __init__(self, experiment: ExperimentConfig, guesser: Guesser, client: MillionaireClient):
        self.experiment = experiment
        self.guesser = guesser
        self.client = client
        self.competitions = [0, 1, 2, 3, 4, 5]

    def run(self, times_per_competition: int = 5, save: bool = True, filename: str = "benchmark_results.xlsx"):
        all_results = []

        for comp_id in self.competitions:
            for game_num in range(times_per_competition):
                try:
                    game = Game(self.client, guesser=self.guesser, competition_id=comp_id)
                    game.play_game()

                    results = game.get_game_results()
                    all_results.extend(results)

                except Exception as e:
                    print(f"Error on game {game_num + 1} of competition {comp_id}: {e}")

        print("Calculating Metrics")
        analysis = Analysis(all_results, self.experiment)
        analysis.calculate_experiments_metrics()

        if save:
            self.save_to_excel(filename)

    def save_to_excel(self, filename: str = "benchmark_results.xlsx"):
        directory = os.path.dirname(filename)
        if directory:
            os.makedirs(directory, exist_ok=True)

        data = dataclasses.asdict(self.experiment)

        if data.get('approach'):
            data['approach'] = data['approach'].value

        data['timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        df_new = pd.DataFrame([data])

        try:
            if os.path.exists(filename):
                df_existing = pd.read_excel(filename)
                df_final = pd.concat([df_existing, df_new], ignore_index=True)
            else:
                df_final = df_new

            df_final.to_excel(filename, index=False)
            print(f"Results successfully saved at: {filename}")
        except Exception as e:
            print(f"Error saving file: {e}")

## Bi- and Cross-Encoder RAG Guesser

This is the implementation of the notebooks approach.

In [ ]:
TOP_K_PAGES = 2
TOP_K_SECTIONS = 15

class CrossEncoderRAGBaseline(Guesser):
    """
    A Guesser using a Retrieve and Re-rank approach.
    It retrieves candidate Wikipedia sections using a fast Bi-Encoder,
    and then uses a Cross-Encoder to score how well the context + option answers the question.
    """
    def __init__(self, config: ExperimentConfig, mode="text"):
        super().__init__(config, mode=mode, transcription_model="base")

        # Bi-Encoder for wikipedia section retrieval
        self.bi_encoder_id = config.embedding_model
        # Cross-Encoder for re-ranking/classification
        self.cross_encoder_id = config.inference_model

        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.bi_encoder = SentenceTransformer(self.bi_encoder_id, device=device, trust_remote_code=True)
        self.cross_encoder = CrossEncoder(self.cross_encoder_id, device=device, trust_remote_code=True)

    def infer_answer(self, question: Question, theme: str, game_session: Any = None) -> int:
        self.search_time = 0.0
        start_time = time.time()

        wiki = wikipediaapi.Wikipedia(user_agent='Polimillionaire', language='en')
        search_start = time.time()
        all_candidate_sections = set()

        # Retrieve wikipedia section for question
        question_title, _ = self._fetch_question_data(wiki, question.text)
        all_candidate_sections.update(self._get_sections_for_query(wiki, question.text, exclude_title=None))

        # Retrieve wikipedia section for options
        for option in question.options:
            all_candidate_sections.update(self._get_sections_for_query(wiki, f"{option.text} - {question_title}", exclude_title=question_title))

        candidate_sections = list(all_candidate_sections)
        self.search_time = time.time() - search_start

        if not candidate_sections:
            # fallback: choose option without rag input
            model_inputs = [(question.text, option.text) for option in question.options]
            scores = self.cross_encoder.predict(model_inputs)
            best_option_idx = np.argmax(scores)
            best_overall_score = scores[best_option_idx]
        else:
            # Bi-Encoder: Find top-k sections that are the most similar to the question
            with torch.no_grad():
                question_emb = self.bi_encoder.encode(question.text, convert_to_tensor=True)
                section_embeddings = self.bi_encoder.encode(candidate_sections, convert_to_tensor=True)
                cos_scores = util.cos_sim(question_emb, section_embeddings)[0]

                top_k = min(TOP_K_SECTIONS, len(candidate_sections))
                top_indices = torch.topk(cos_scores, k=top_k).indices.cpu().numpy()
                top_sections = [candidate_sections[i] for i in top_indices]

            # Cross-Encoder: Find the best match between the question and (option + section)
            all_pairs = []
            for option in question.options:
                for section in top_sections:
                    all_pairs.append((question.text, f"{option.text}. {section}"))

            print(f"[Cross-RAG] Batch predicting {len(all_pairs)} pairs...")
            all_scores = self.cross_encoder.predict(all_pairs, batch_size=6)
            all_scores = all_scores.reshape(len(question.options), len(top_sections))
            option_max_scores = np.max(all_scores, axis=1)

            for opt_idx, score in enumerate(option_max_scores):
                print(f"  Option {opt_idx} ({question.options[opt_idx].text[:20]}...): Score = {score:.4f}")

            # select best option match
            best_option_idx = np.argmax(option_max_scores)
            best_overall_score = option_max_scores[best_option_idx]

        self.reasoning_time = time.time() - start_time - self.search_time

        print(f"[Cross-RAG] Search & Reasoning took {self.search_time + self.reasoning_time:.2f}s")
        print(f"[Cross-RAG] Selected Option {best_option_idx} with Score: {best_overall_score:.4f}")

        return question.options[best_option_idx].id

    def _fetch_question_data(self, wiki: wikipediaapi.Wikipedia, text: str) -> tuple[str, str]:
        """Fetches the top article title and summary for the question."""
        try:
            results = wiki.search(text)
            if results.pages:
                title = list(results.pages.keys())[0]
                return title, results.pages[title].summary
        except Exception as e:
            print(f"[Cross-RAG] Error searching for question: {e}")
        return "", ""

    def _get_sections_for_query(self, wiki: wikipediaapi.Wikipedia, query: str, exclude_title: str = None) -> list[str]:
        """Retrieves sections from Wikipedia for a given query for the top-k pages."""
        try:
            results = wiki.search(query)
            if not results.pages:
                return []

            top_titles = []
            for title in results.pages.keys():
                if title != exclude_title:
                    top_titles.append(title)
                if len(top_titles) == TOP_K_PAGES:
                    break

            sections = []
            for title in top_titles:
                page = results.pages[title]
                sections.extend(self._get_page_sections(page))
            return sections
        except Exception as e:
            print(f"[Cross-RAG] Error fetching sections for '{query}': {e}")
            return []

    def _get_page_sections(self, page: wikipediaapi.WikipediaPage) -> list[str]:
        """Extracts clean text sections from a Wikipedia page."""
        sections_text = [page.summary]
        exclude = {"See also", "References", "Further reading", "External links", "Notes", "Sources", "Bibliography", "Gallery"}

        for s in page.sections:
            if s.title not in exclude and s.text.strip():
                sections_text.append(s.text)
        return sections_text

## Run Experiment

Run benchmarked experiment with multiple attempts per competition to determine accuracy. First `text` then `speech` mode.

In [ ]:
ATTEMPTS_PER_COMPETITION = 10
API_URL = "http://131.175.15.22:51111/"

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
USERNAME = userdata.get('POLI_USERNAME')
PASSWORD = userdata.get('POLI_PASSWORD')

client = MillionaireClient(API_URL)
client.login(USERNAME, PASSWORD)

config = ExperimentConfig(
        username="Luca",
        notes="Retrieve & Re-rank RAG (Bi-Encoder + Cross-Encoder)",
        approach=ApproachType.RAG,
        is_rag=True,
        embedding_model="nomic-ai/nomic-embed-text-v1.5",
        embedding_model_size=0.1, # 0.1B
        inference_model="cross-encoder/ettin-reranker-1b-v1",
        inference_model_size=1.0, # 1B
    )

guesser = CrossEncoderRAGBaseline(config, mode="text")
benchmark = Benchmark(config, guesser, client)
benchmark.run(times_per_competition=ATTEMPTS_PER_COMPETITION, filename="bi_cross_encoder_text.xlsx")

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

4_Dense/model.safetensors:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

Preloading models...
Started game in competition: Entertainment (Mode: text)

--- Level 1 ---
Q: How does the film 'Lawrence of Arabia' relate to the historical figure T. E. Lawrence?
  [0] It is a fictional story with no basis in reality
  [1] It is a dramatized version of Lawrence's life and experiences
  [2] It focuses solely on Lawrence's archaeological work
  [3] It accurately portrays all of Lawrence's actions and decisions

Guesser is thinking...


[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


[Cross-RAG] Batch predicting 60 pairs...
  Option 0 (It is a fictional st...): Score = 10.1285
  Option 1 (It is a dramatized v...): Score = 10.2968
  Option 2 (It focuses solely on...): Score = 9.8784
  Option 3 (It accurately portra...): Score = 9.9655
[Cross-RAG] Search & Reasoning took 12.05s
[Cross-RAG] Selected Option 1 with Score: 10.2968
Guesser chose option: 1 (Time: 12.05s, Transcription: 0.00s, Search: 9.44s, Reasoning: 2.61s)
 CORRECT!

--- Level 2 ---
Q: What term describes a song specially created for an anime series?
  [0] Anime theme song
  [1] Cinematic score
  [2] Pop ballad
  [3] Orchestral piece

Guesser is thinking...
[Cross-RAG] Batch predicting 60 pairs...
  Option 0 (Anime theme song...): Score = 8.2879
  Option 1 (Cinematic score...): Score = 6.1010
  Option 2 (Pop ballad...): Score = 5.4681
  Option 3 (Orchestral piece...): Score = 5.8354
[Cross-RAG] Search & Reasoning took 9.40s
[Cross-RAG] Selected Option 0 with Score: 8.2879
Guesser chose option: 0 (Time: 9

In [ ]:
ATTEMPTS_PER_COMPETITION = 5
API_URL = "http://131.175.15.22:51111/"

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
USERNAME = userdata.get('POLI_USERNAME')
PASSWORD = userdata.get('POLI_PASSWORD')

client = MillionaireClient(API_URL)
client.login(USERNAME, PASSWORD)

config = ExperimentConfig(
        username="Luca",
        notes="Retrieve & Re-rank RAG (Bi-Encoder + Cross-Encoder)",
        approach=ApproachType.RAG,
        is_rag=True,
        embedding_model="nomic-ai/nomic-embed-text-v1.5",
        embedding_model_size=0.1, # 0.1B
        inference_model="cross-encoder/ettin-reranker-1b-v1",
        inference_model_size=1.0, # 1B
    )

guesser = CrossEncoderRAGBaseline(config, mode="speech")
benchmark_speech = Benchmark(config, guesser, client)
benchmark_speech.run(times_per_competition=ATTEMPTS_PER_COMPETITION, filename="bi_cross_encoder_speech.xlsx")

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Preloading models...
Loading Whisper 'base' model...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 206MiB/s]


Whisper 'base' model preloaded and ready.
Started game in competition: Entertainment (Mode: speech)

--- Level 1 ---
Speech mode active: Fetching and transcribing audio...
  Waiting for transcriptions to finish...
Q: How does the concept of absence as a lyrical theme manifest in Pink Floyd's music? Particularly in the song, have a cigar from Wish You Were Here
  [0] By portraying the absence of a person or feeling through metaphor and suggestion
  [1] Through the direct mention of missing loved ones
  [2] Haha! Through the explicit use of space and silence in the song structure
  [3] by focusing on the physical absence of band members during performances

Guesser is thinking...
[Cross-RAG] Batch predicting 60 pairs...
  Option 0 (By portraying the ab...): Score = 6.2511
  Option 1 (Through the direct m...): Score = 4.1917
  Option 2 (Haha! Through the ex...): Score = 3.1580
  Option 3 (by focusing on the p...): Score = 5.7027
[Cross-RAG] Search & Reasoning took 17.29s
[Cross-RAG] Selec